In [1]:
from HypEx.hypex.dataset import (
    Dataset,
    InfoRole,
    StratificationRole,
    TargetRole,
    TreatmentRole,
)
from HypEx.hypex.utils.enums import BackendsEnum
from pyspark.sql import SparkSession
import pyspark.sql as spark
import pandas as pd

In [2]:
session = (
            SparkSession.builder
            .master("local[*]")
            .config("spark.driver.memory", "4g")
            .config("spark.executo|r.memory", "4g")
            .config("spark.memory.fraction", "0.8") 
            .config("spark.memory.storageFraction", "0.3") 
            .getOrCreate()
          )

26/03/11 11:25:56 WARN Utils: Your hostname, eric-Katana-17-B12UCR resolves to a loopback address: 127.0.1.1; using 10.240.72.74 instead (on interface wlo1)
26/03/11 11:25:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 11:25:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
data = (
            session.read.format('csv')
            .option("header", "true")
            .option("inferSchema", "true")
            .option("ignoreLeadingWhiteSpace", "true") 
            .option("ignoreTrailingWhiteSpace", "true")
            .load("./data_no_index.csv")
        )
data.show()

+-------+------------+-----+----------+------------------+----+------+----------+
|user_id|signup_month|treat|pre_spends|       post_spends| age|gender|  industry|
+-------+------------+-----+----------+------------------+----+------+----------+
|    0.0|         0.0|  0.0|     519.5| 424.6666666666667|39.0|     M| Logistics|
|    1.0|         0.0|  0.0|     500.5| 423.1111111111111|19.0|     F| Logistics|
|    2.0|         0.0|  0.0|     499.0|423.44444444444446|29.0|     M|E-commerce|
|    3.0|         2.0|  1.0|     494.0| 530.5555555555555|38.0|     M|E-commerce|
|    4.0|         9.0|  1.0|     455.0| 451.8888888888889|53.0|     F| Logistics|
|    5.0|         1.0|  1.0|     536.0| 518.3333333333334|22.0|     F| Logistics|
|    6.0|         4.0|  1.0|     493.5|500.77777777777777|21.0|     M|E-commerce|
|    7.0|         8.0|  1.0|     484.0| 447.3333333333333|69.0|     F|E-commerce|
|    8.0|         0.0|  0.0|     473.5|413.22222222222223|34.0|     M|E-commerce|
|    9.0|       

In [4]:
spark_ds = Dataset(roles={
        "user_id": InfoRole(int),
        "treat": TreatmentRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str)
    }, 
    data=data, 
    session=session,
    backend=BackendsEnum.spark)

In [7]:
spark_ds

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0,0.0,0,519.5,424.666667,39.0,M,Logistics
1,1,0.0,0,500.5,423.111111,19.0,F,Logistics
2,2,0.0,0,499.0,423.444444,29.0,M,E-commerce
3,3,2.0,1,494.0,530.555556,38.0,M,E-commerce
4,4,9.0,1,455.0,451.888889,53.0,F,Logistics
...,...,...,...,...,...,...,...,...
9995,9995,0.0,0,490.0,429.555556,26.0,F,Logistics
9996,9996,0.0,0,464.5,414.666667,49.0,M,E-commerce
9997,9997,0.0,0,498.0,411.666667,30.0,F,E-commerce
9998,9998,2.0,1,528.0,516.555556,36.0,F,Logistics


In [5]:
ds = Dataset(roles={
        "user_id": InfoRole(int),
        "treat": TreatmentRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str)
    }, 
    data=data, 
    backend=BackendsEnum.pandas)

In [6]:
ds

,user_id,signup_month,treat,pre_spends,post_spends,age,gender,industry
0,0.0,0.0,0.0,519.5,424.666656,39.0,1,Logistics
1,1.0,0.0,0.0,500.5,423.111115,19.0,0,Logistics
2,2.0,0.0,0.0,499.0,423.444458,29.0,1,E-commerce
3,3.0,2.0,1.0,494.0,530.555542,38.0,1,E-commerce
4,4.0,9.0,1.0,455.0,451.888885,53.0,0,Logistics
...,...,...,...,...,...,...,...,...
9995,9995.0,0.0,0.0,490.0,429.555542,26.0,0,Logistics
9996,9996.0,0.0,0.0,464.5,414.666656,49.0,1,E-commerce
9997,9997.0,0.0,0.0,498.0,411.666656,30.0,0,E-commerce
9998,9998.0,2.0,1.0,528.0,516.555542,36.0,0,Logistics


In [31]:
session.stop()